In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install haversine

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from haversine import haversine, Unit

## Setting
# Load file path
candidate_file_path = '/content/drive/MyDrive/Project2_K-means algorithm/Data_vertiport_candidates.xlsx'
traffic_file_path = '/content/drive/MyDrive/Project2_K-means algorithm/Traffic_Data.csv'
hospital_file_path = '/content/drive/MyDrive/Project2_K-means algorithm/General_Hospitals_Coordinates.csv'
tour_file_path = '/content/drive/MyDrive/Project2_K-means algorithm/Tourist_Attraction_Data.csv'

# Read the data
candidate_df = pd.read_excel(candidate_file_path)

try:
    traffic_df = pd.read_csv(traffic_file_path, encoding='utf-8-sig')
except UnicodeDecodeError:
    traffic_df = pd.read_csv(traffic_file_path, encoding='cp949')

hospital_df = pd.read_csv(hospital_file_path)
tour_df = pd.read_csv(tour_file_path)

# Remove empty space
candidate_df.columns = candidate_df.columns.str.strip()
traffic_df.columns = traffic_df.columns.str.strip()
hospital_df.columns = hospital_df.columns.str.strip()
tour_df.columns = tour_df.columns.str.strip()

# Remove duplication
candidate_df = candidate_df.drop_duplicates(subset=['Latitude (deg)', 'Longitude (deg)'])
traffic_df = traffic_df.drop_duplicates(subset=['latitude', 'longitude', 'traffic'])
hospital_df = hospital_df.drop_duplicates(subset=['위도', '경도'])
tour_df = tour_df.drop_duplicates(subset=['위도', '경도'])

# Remove ',' and change
traffic_df['traffic'] = traffic_df['traffic'].astype(str).str.replace(',', '').str.strip().astype(float)

# Remove NAN
candidate_df = candidate_df.dropna()
traffic_df = traffic_df.dropna(subset=['longitude', 'latitude', 'traffic'])
hospital_df = hospital_df.dropna()
tour_df = tour_df.dropna()

# check
print(len(candidate_df))
print(candidate_df['Latitude (deg)'].head())
print(f"{candidate_df['Latitude (deg)'][0]:.15f}")

958
0    37.621876
1    37.535756
2    37.492020
3    37.612760
4    37.561111
Name: Latitude (deg), dtype: float64
37.621875877962800


In [ ]:
# train data
sampled_traffic = traffic_df.sample(n=500, random_state=42)

# separate values
X_train = sampled_traffic[['longitude', 'latitude']]
y_train = sampled_traffic['traffic']

# make a regrression model(made by Gemini)
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# apply to vertiport candidate data
point_test = candidate_df[['Longitude (deg)', 'Latitude (deg)']]

point_test.columns = ['longitude', 'latitude']

# predict the traffic on vertiport cadidate location by regression model
predicted_traffic = model.predict(point_test)

# rescaling 0~1
traffic_score = predicted_traffic / max(predicted_traffic)

candidate_df['Traffic_Score'] = traffic_score

In [ ]:
# convert the location of hospitals to tuple
hospital_coords = list(zip(hospital_df['위도'], hospital_df['경도']))

# set score list
hospital_score = []

# calculate the distance from each candidate location to hospitals
for idx, row in candidate_df.iterrows():
    cand_lat = row['Latitude (deg)']
    cand_lon = row['Longitude (deg)']
    cand_coord = (cand_lat, cand_lon)

    total_bonus = 0


    for hos_coord in hospital_coords:
        # To kilometers by haversine
        dist = haversine(cand_coord, hos_coord, unit=Unit.KILOMETERS)

        # distance limit
        if dist <= 3.0:
            # the mechanism to evaluate the score(linear line)
            score = 1 - (dist / 3.0)

            # Gain the score at a point(location of vertiport candidate).
            total_bonus += score

    # Add to list
    hospital_score.append(total_bonus)

# Add the hospital score data to given original file
candidate_df['Hospital_Score'] = hospital_score

# scaling 0~1
max_score = candidate_df['Hospital_Score'].max()


if max_score > 0:
    candidate_df['Hospital_Score'] = candidate_df['Hospital_Score'] / max_score

In [ ]:
# convert the location of tourist attractions to tuple
tour_coords = list(zip(tour_df['위도'], tour_df['경도']))

# set score list
tour_score = []

# calculate the distance from each candidate location to tourist attractions
for idx, row in candidate_df.iterrows():
    cand_lat = row['Latitude (deg)']
    cand_lon = row['Longitude (deg)']
    cand_coord = (cand_lat, cand_lon)

    total_bonus = 0


    for tour_coord in tour_coords:
        # To kilometers by haversine
        dist = haversine(cand_coord, tour_coord, unit=Unit.KILOMETERS)

        # distance limit
        if dist <= 3.0:
            # the mechanism to evaluate the score(linear line)
            score = 1 - (dist / 3.0)

            # Gain the score at a point(location of vertiport candidate).
            total_bonus += score

    # Add to list
    tour_score.append(total_bonus)

# Add the tour score data to given original file
candidate_df['Tour_Score'] = tour_score

# scaling 0~1
max_score = candidate_df['Tour_Score'].max()


if max_score > 0:
    candidate_df['Tour_Score'] = candidate_df['Tour_Score'] / max_score

# Checking the results
print(candidate_df[['Longitude (deg)', 'Latitude (deg)', 'Traffic_Score','Hospital_Score', 'Tour_Score']].head(10))

   Longitude (deg)  Latitude (deg)  Traffic_Score  Hospital_Score  Tour_Score
0       126.940197       37.621876       0.634934        0.142848    0.039885
1       127.096292       37.535756       0.716298        0.444204    0.000000
2       126.939432       37.492020       0.714025        0.482551    0.003242
3       127.033749       37.612760       0.649670        0.120674    0.000000
4       126.839673       37.561111       0.725008        0.739356    0.165392
5       127.062331       37.468052       0.714367        0.009125    0.036120
6       126.989846       37.548700       0.720113        0.194858    0.493848
7       127.135744       37.490100       1.000000        0.176205    0.000000
8       126.882609       37.497940       0.740606        0.862753    0.000000
9       126.990198       37.495474       0.719748        0.258489    0.068291


In [ ]:
# Set weight parameters(w1+w2+w3=5)
w1=1.5
w2=2
w3=1.5

# Reduce candidates
candidate_df['Total_Score'] = w1*candidate_df['Traffic_Score'] + w2*candidate_df['Hospital_Score'] + w3*candidate_df['Tour_Score']
sorted_df = candidate_df.sort_values(by='Total_Score', ascending=False)

# Set the path to save
save_path = '/content/drive/MyDrive/Project2_K-means algorithm/Processed_Data_vertiport_candidates_scores.csv'

# Save the csv.file in google drive
candidate_df.to_csv(save_path, index=False, encoding='utf-8-sig')

# Select 1000 data
top_400 = sorted_df.head(400)

# Save the file
top_400.to_csv('/content/drive/MyDrive/Project2_K-means algorithm/Top_400_Candidates.csv', index=False, encoding='utf-8-sig')

In [ ]:
for iteration in range(max_iter):

    iterations_taken = iteration + 1


    clusters = []

# Assign points to nearest centroid
    for point in points:

        # Calculate distance from point to each centroid
        distances = np.linalg.norm(point - centroids, axis=1)

        # Find nearest centroid index
        cluster = np.argmin(distances)

        clusters.append(cluster)

    clusters = np.array(clusters)

    # Update centroids by calculating mean of points in each cluster
    new_centroids = []

    for i in range(K):

        # Get all points in cluster i
        cluster_points = points[clusters == i]

        # Keep the centroid if a cluster becomes empty
        if len(cluster_points) == 0:
            new_centroid = centroids[i]
        else:
            # Calculate mean
            new_centroid = cluster_points.mean(axis=0)

        new_centroids.append(new_centroid)

    new_centroids = np.array(new_centroids)

    # Stop when centroids stop moving meaningfully
    if np.allclose(centroids, new_centroids, atol=tol):
        centroids = new_centroids
        break

    # Update centroid values
    centroids = new_centroids
